# Vector Add 教学版

这个 notebook 用来逐格讲清楚最小闭环：定义计算、lower 成 PrimFunc、编译、执行、验证。

## 运行口径

- 当前环境优先跑 CPU fallback
- 如果后续切到 GPU，需要可用 CUDA 版 TVM 和 NVIDIA 驱动
- `.py` 版本仍然是权威实现，这个 notebook 负责教学展示

In [ ]:
import numpy as np
import tvm
from tvm import te
from tvm.ir import IRModule
from tvm.runtime._tensor import tensor as tvm_tensor

In [ ]:
n = 1024
A = te.placeholder((n,), name="A", dtype="float32")
B = te.placeholder((n,), name="B", dtype="float32")
C = te.compute((n,), lambda i: A[i] + B[i], name="C")

prim_func = te.create_prim_func([A, B, C])
mod = IRModule({"vector_add": prim_func})
f = tvm.build(mod, target="llvm")
print(f)

In [ ]:
a_np = np.random.uniform(0, 1, (n,)).astype("float32")
b_np = np.random.uniform(0, 1, (n,)).astype("float32")

a = tvm_tensor(a_np, device=tvm.cpu(0))
b = tvm_tensor(b_np, device=tvm.cpu(0))
c = tvm_tensor(np.zeros((n,), dtype="float32"), device=tvm.cpu(0))

f(a, b, c)
error = np.max(np.abs(c.numpy() - (a_np + b_np)))
print(f"max error: {error:.2e}")